In [40]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Step 1: Create an imbalanced binary classification dataset
X, y = make_classification(n_samples=1000, n_features=10, n_informative=2, n_redundant=8, 
                           weights=[0.9, 0.1], flip_y=0, random_state=42)

np.unique(y, return_counts=True)

(array([0, 1]), array([90, 10]))

In [ ]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

In [43]:
# Define the model hyperparameters
# params = {
#     "solver": "lbfgs",
#     "max_iter": 1000,
#     "multi_class": "auto",
#     "random_state": 8888,
# }

params = {
    "solver": "lbfgs",
    "max_iter": 1000,
    "random_state": 8888,
}

# Train the model
lr = LogisticRegression(**params)
lr.fit(X_train, y_train)

# Predict on the test set
y_pred = lr.predict(X_test)

report = classification_report(y_test, y_pred)
print(report)

              precision    recall  f1-score   support

           0       0.95      0.98      0.96        81
           1       0.71      0.56      0.62         9

    accuracy                           0.93        90
   macro avg       0.83      0.77      0.79        90
weighted avg       0.93      0.93      0.93        90



In [44]:
report_dict = classification_report(y_test, y_pred, output_dict=True)
report_dict

{'0': {'precision': 0.9518072289156626,
  'recall': 0.9753086419753086,
  'f1-score': 0.9634146341463414,
  'support': 81.0},
 '1': {'precision': 0.7142857142857143,
  'recall': 0.5555555555555556,
  'f1-score': 0.625,
  'support': 9.0},
 'accuracy': 0.9333333333333333,
 'macro avg': {'precision': 0.8330464716006885,
  'recall': 0.7654320987654322,
  'f1-score': 0.7942073170731707,
  'support': 90.0},
 'weighted avg': {'precision': 0.9280550774526678,
  'recall': 0.9333333333333333,
  'f1-score': 0.9295731707317073,
  'support': 90.0}}

In [45]:
import mlflow

In [46]:
mlflow.set_experiment("First Experiment")
mlflow.set_tracking_uri(uri="http://127.0.0.1:5000/")

with mlflow.start_run():
    mlflow.log_params(params)
    mlflow.log_metrics({
        'accuracy': report_dict['accuracy'],
        'recall_class_0': report_dict['0']['recall'],
        'recall_class_1': report_dict['1']['recall'],
        'f1_score_macro': report_dict['macro avg']['f1-score']
    })
    mlflow.sklearn.log_model(lr, "Logistic Regression")  

2026/04/20 03:57:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/20 03:57:12 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run youthful-turtle-843 at: http://127.0.0.1:5000/#/experiments/1/runs/1b5a3f97e91b49468d641e7dee8945b4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [47]:
# Debug helper: show latest runs in this experiment
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri("http://127.0.0.1:5000/")
client = MlflowClient()
exp = client.get_experiment_by_name("First Experiment")

if exp is None:
    print("Experiment 'First Experiment' not found")
else:
    print(f"Experiment ID: {exp.experiment_id}")
    runs = client.search_runs(
        [exp.experiment_id],
        order_by=["attributes.start_time DESC"],
        max_results=5,
    )
    print(f"Latest runs found: {len(runs)}")
    for r in runs:
        print(
            f"run_id={r.info.run_id} | status={r.info.status} | metrics={sorted(r.data.metrics.keys())}"
        )

Experiment ID: 1
Latest runs found: 5
run_id=1b5a3f97e91b49468d641e7dee8945b4 | status=FINISHED | metrics=['accuracy', 'f1_score_macro', 'recall_class_0', 'recall_class_1']
run_id=6393335f3bb64d7b8cb50e2a737cf711 | status=FINISHED | metrics=['accuracy', 'f1_score_macro', 'recall_class_0', 'recall_class_1']
run_id=23c4870a281f43738157f4f6795c5a8b | status=FINISHED | metrics=['accuracy', 'f1_score_macro', 'recall_class_0', 'recall_class_1']
run_id=5850ff9ac7334da79b0609073b7b0d34 | status=FINISHED | metrics=['accuracy', 'f1_score_macro', 'recall_class_0', 'recall_class_1']
run_id=ba80c76d8db5416593f731869f9dce0d | status=FINISHED | metrics=['accuracy', 'f1_score_macro', 'recall_class_0', 'recall_class_1']
